In [10]:
# ==============================
# IMPORT DES LIBRAIRIES
# ==============================
import pandas as pd


In [11]:
# ==============================
# CHARGEMENT DU DATASET BRUT
# ==============================
# On charge le fichier CSV depuis le dossier raw
df = pd.read_csv("../data/raw/annecy_matches_2025_2026.tsv", sep="\t")

# ==============================
# INFORMATIONS SUR LE DATASET
# ==============================

# Permet de voir :
# - le nombre de lignes
# - les types de données
# - les valeurs manquantes
df.info()
print("\n--------------------------------")
print("Nombre de lignes et de colonnes :", df.shape)



<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            26 non-null     str    
 1   Time            26 non-null     str    
 2   Round           26 non-null     str    
 3   Day             26 non-null     str    
 4   Venue           26 non-null     str    
 5   Result          26 non-null     str    
 6   GF              26 non-null     int64  
 7   GA              26 non-null     int64  
 8   Opponent        26 non-null     str    
 9   Poss            26 non-null     int64  
 10  Attendance      24 non-null     str    
 11  Captain         26 non-null     str    
 12  Formation       26 non-null     str    
 13  Opp Formation   26 non-null     str    
 14  Referee         26 non-null     str    
 15  Match Report    26 non-null     str    
 16  Notes           0 non-null      float64
dtypes: float64(1), int64(3), str(13)
memory usage: 3

In [12]:
# ==============================
# NETTOYAGE DES NOMS DE COLONNES
# ==============================
# Suppression des espaces inutiles avant/après les noms de colonnes
# Cela évite les erreurs lors de l'accès aux colonnes (ex : " Date" vs "Date")

df.columns = df.columns.str.strip()

# Vérification des noms de colonnes
print(df.columns)

Index(['Date', 'Time', 'Round', 'Day', 'Venue', 'Result', 'GF', 'GA',
       'Opponent', 'Poss', 'Attendance', 'Captain', 'Formation',
       'Opp Formation', 'Referee', 'Match Report', 'Notes'],
      dtype='str')


In [13]:
# ==============================
# SUPPRESSION DES COLONNES INUTILES
# ==============================
# Certaines colonnes ne sont pas utiles pour l'analyse
# errors="ignore" évite une erreur si la colonne n'existe pas déjà
df = df.drop(columns=[
    "Match Report",
    "Notes",
], errors="ignore")

# ==============================
# CONVERSION DE LA DATE
# ==============================
# La colonne Date est actuellement du texte
# On la convertit en vrai format datetime
df["Date"] = pd.to_datetime(df["Date"])


# ==============================
# NETTOYAGE DE LA COLONNE ATTENDANCE
# ==============================
# Attendance = nombre de spectateurs présents au stade
# Exemple : "2,901"

# On enlève la virgule
df["Attendance"] = df["Attendance"].astype(str).str.replace(",", "")

# On convertit en nombre
df["Attendance"] = pd.to_numeric(df["Attendance"], errors="coerce")

# ==============================
# VÉRIFICATION DES TYPES APRÈS NETTOYAGE
# ==============================
df.info()
print("\n--------------------------------")
print("Nombre de lignes et de colonnes :", df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           26 non-null     datetime64[us]
 1   Time           26 non-null     str           
 2   Round          26 non-null     str           
 3   Day            26 non-null     str           
 4   Venue          26 non-null     str           
 5   Result         26 non-null     str           
 6   GF             26 non-null     int64         
 7   GA             26 non-null     int64         
 8   Opponent       26 non-null     str           
 9   Poss           26 non-null     int64         
 10  Attendance     24 non-null     float64       
 11  Captain        26 non-null     str           
 12  Formation      26 non-null     str           
 13  Opp Formation  26 non-null     str           
 14  Referee        26 non-null     str           
dtypes: datetime64[us](1), float64(1), in

In [18]:
# ==============================
# FEATURE ENGINEERING
# ==============================

# Trier les matchs par date
df = df.sort_values("Date")

# Réinitialiser l'index
df = df.reset_index(drop=True)

# Différence de buts
df["Goal_Diff"] = df["GF"] - df["GA"]

# Points obtenus selon le résultat
df["Points"] = df["Result"].map({
    "W": 3,
    "D": 1,
    "L": 0
})

# Indicateur domicile / extérieur
df["Is_Home"] = df["Venue"].map({
    "Home": 1,
    "Away": 0
})

# Numéro de journée
df["Matchweek"] = df["Round"].str.extract(r'(\d+)').astype(int)

# Points cumulés dans la saison
df["Points_Cum"] = df["Points"].cumsum()

# Différence de buts cumulée
df["Goal_Diff_Cum"] = df["Goal_Diff"].cumsum()

# Aperçu
df.head()

,Date,Time,Round,Day,Venue,Result,GF,GA,Opponent,Poss,...,Captain,Formation,Opp Formation,Referee,Goal_Diff,Points,Is_Home,Points_Cum,Goal_Diff_Cum,Matchweek
0,2025-08-09,20:00,Matchweek 1,Sat,Away,L,0,2,Pau FC,42,...,Ahmed Kashi,4-3-3,4-1-4-1,Tifenn Leprodhomme,-2,0,0,0,-2,1
1,2025-08-16,14:00,Matchweek 2,Sat,Home,D,2,2,USL Dunkerque,37,...,Ahmed Kashi,3-4-3,4-1-4-1,Faouzi Benchabane,0,1,1,1,-2,2
2,2025-08-22,20:00,Matchweek 3,Fri,Away,W,1,0,Amiens,35,...,Ahmed Kashi,3-4-3,4-4-2,Geoffrey Kubler,1,3,0,4,-1,3
3,2025-08-29,20:00,Matchweek 4,Fri,Away,L,0,1,Red Star,62,...,Ahmed Kashi,3-4-3,3-4-1-2,Mikael Lesage,-1,0,0,4,-2,4
4,2025-09-13,14:00,Matchweek 5,Sat,Home,D,1,1,Reims,32,...,Ahmed Kashi,3-4-3,4-2-3-1,Mathieu Vernice,0,1,1,5,-2,5


In [ ]:
# ==============================
# SAUVEGARDE DU DATASET PRÉPARÉ
# ==============================
# On enregistre la version enrichie du dataset dans le dossier processed

df.to_csv("../data/processed/annecy_matches_features.csv", index=False)

,Date,Time,Round,Day,Venue,Result,GF,GA,Opponent,Poss,...,Captain,Formation,Opp Formation,Referee,Goal_Diff,Points,Is_Home,Points_Cum,Goal_Diff_Cum,Matchweek
0,2025-08-09,20:00,Matchweek 1,Sat,Away,L,0,2,Pau FC,42,...,Ahmed Kashi,4-3-3,4-1-4-1,Tifenn Leprodhomme,-2,0,0,0,-2,1
1,2025-08-16,14:00,Matchweek 2,Sat,Home,D,2,2,USL Dunkerque,37,...,Ahmed Kashi,3-4-3,4-1-4-1,Faouzi Benchabane,0,1,1,1,-2,2
2,2025-08-22,20:00,Matchweek 3,Fri,Away,W,1,0,Amiens,35,...,Ahmed Kashi,3-4-3,4-4-2,Geoffrey Kubler,1,3,0,4,-1,3
3,2025-08-29,20:00,Matchweek 4,Fri,Away,L,0,1,Red Star,62,...,Ahmed Kashi,3-4-3,3-4-1-2,Mikael Lesage,-1,0,0,4,-2,4
4,2025-09-13,14:00,Matchweek 5,Sat,Home,D,1,1,Reims,32,...,Ahmed Kashi,3-4-3,4-2-3-1,Mathieu Vernice,0,1,1,5,-2,5
